<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/Sleep%20Staging%20Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/sample_submission.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00171.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00802.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00442.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00109.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00709.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00722.csv
/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/test001/test001_00425.cs

# Setup

In [ ]:
print('Setup complete')


Setup complete


# Import and Config

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')
TRAIN_DIRNAME = 'train'
TEST_SEGMENT_DIRNAME = 'test_segment'
TARGET = 'Sleep_Stage'
SEGMENT_SIZE = 16 * 30
SEED = 42

RAW_FEATURES = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']
print('Config ready')


Config ready


# Load Data

In [ ]:
def find_file(filename: str):
    matches = list(INPUT_ROOT.rglob(filename))
    return matches[0] if matches else None

def find_named_dirs(dirname: str):
    return [p for p in INPUT_ROOT.rglob('*') if p.is_dir() and p.name == dirname]

sample_sub_path = find_file('sample_submission.csv')
train_dirs = find_named_dirs(TRAIN_DIRNAME)
test_segment_dirs = find_named_dirs(TEST_SEGMENT_DIRNAME)

print('sample_sub_path =', sample_sub_path)
print('train_dirs =', train_dirs[:10])
print('test_segment_dirs =', test_segment_dirs[:10])

if sample_sub_path is None:
    raise FileNotFoundError('sample_submission.csv not found')

# เลือกโฟลเดอร์ train/test_segment ที่มีไฟล์ csv อยู่จริง
train_root = None
for d in train_dirs:
    if list(d.glob('*.csv')):
        train_root = d
        break

test_root = None
for d in test_segment_dirs:
    if list(d.glob('*/*.csv')) or list(d.glob('*.csv')):
        test_root = d
        break

print('train_root =', train_root)
print('test_root =', test_root)

if train_root is None or test_root is None:
    raise FileNotFoundError('train root or test_segment root not found')

sample_sub = pd.read_csv(sample_sub_path)
display(sample_sub.head(10))


sample_sub_path = /kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/sample_submission.csv
train_dirs = [PosixPath('/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/train'), PosixPath('/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/train/train')]
test_segment_dirs = [PosixPath('/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment'), PosixPath('/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment')]
train_root = /kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/train/train
test_root = /kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment


,id,labels
0,test001_00000,W
1,test001_00001,W
2,test001_00002,W
3,test001_00003,NaN
4,test001_00004,NaN
5,test001_00005,NaN
6,test001_00006,NaN
7,test001_00007,NaN
8,test001_00008,NaN
9,test001_00009,NaN


# Feature Engineering

In [ ]:
def add_signal_features(df):
    df = df.copy()
    df['ACC_MAG'] = np.sqrt(df['ACC_X']**2 + df['ACC_Y']**2 + df['ACC_Z']**2)
    return df

def summarize_segment(df, feature_cols):
    row = {}
    for col in feature_cols:
        s = pd.to_numeric(df[col], errors='coerce')
        row[f'{col}_mean'] = s.mean()
        row[f'{col}_std'] = s.std()
        row[f'{col}_min'] = s.min()
        row[f'{col}_max'] = s.max()
        row[f'{col}_median'] = s.median()
        row[f'{col}_q25'] = s.quantile(0.25)
        row[f'{col}_q75'] = s.quantile(0.75)
        row[f'{col}_range'] = s.max() - s.min()

        diff = s.diff().dropna()
        row[f'{col}_diff_mean'] = diff.mean() if len(diff) else 0.0
        row[f'{col}_diff_std'] = diff.std() if len(diff) else 0.0

    return row

FEATURE_COLS = RAW_FEATURES + ['ACC_MAG']
print('Feature cols =', FEATURE_COLS)


Feature cols = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI', 'ACC_MAG']


# Build Train Segments

In [ ]:
def build_train_table(train_root: Path):
    rows = []
    train_files = sorted(train_root.glob('*.csv'))
    print('num train files =', len(train_files))

    for file_path in train_files:
        df = pd.read_csv(file_path)
        df = add_signal_features(df)
        subject_id = file_path.stem

        n = len(df)
        n_segments = n // SEGMENT_SIZE

        for seg_idx in range(n_segments):
            start = seg_idx * SEGMENT_SIZE
            end = start + SEGMENT_SIZE
            seg = df.iloc[start:end].copy()

            label = seg[TARGET].mode().iloc[0]
            feat = summarize_segment(seg, FEATURE_COLS)
            feat['subject_id'] = subject_id
            feat['segment_id'] = f'{subject_id}_{seg_idx:05d}'
            feat[TARGET] = label
            rows.append(feat)

    return pd.DataFrame(rows)

train_table = build_train_table(train_root)
print('train_table shape =', train_table.shape)
display(train_table.head())
print(train_table[TARGET].value_counts())


num train files = 83
train_table shape = (66745, 93)


,BVP_mean,BVP_std,BVP_min,BVP_max,BVP_median,BVP_q25,BVP_q75,BVP_range,BVP_diff_mean,BVP_diff_std,...,ACC_MAG_max,ACC_MAG_median,ACC_MAG_q25,ACC_MAG_q75,ACC_MAG_range,ACC_MAG_diff_mean,ACC_MAG_diff_std,subject_id,segment_id,Sleep_Stage
0,2.816168,156.050937,-1212.411684,665.980817,1.940699,-11.156508,34.338479,1878.392500,0.170899,94.270326,...,74.209121,64.458164,64.191478,64.735390,16.390483,0.000932,1.180524,train001,train001_00000,W
1,-2.661875,123.266837,-729.854625,466.121232,1.047080,-28.886623,38.463120,1195.975857,-0.120221,59.875134,...,65.868871,64.500011,64.311746,64.699519,2.762024,-0.000096,0.411606,train001,train001_00001,W
2,-0.244761,52.941762,-184.643390,154.234163,2.691504,-22.407477,27.584063,338.877553,-0.121613,31.812076,...,65.336618,64.447980,64.304465,64.610608,1.887605,0.001143,0.268786,train001,train001_00002,W
3,0.004561,76.620256,-344.279599,229.873620,5.574753,-37.435514,44.803622,574.153219,0.001381,42.927390,...,65.387643,64.508938,64.345421,64.657904,1.834768,-0.000563,0.298507,train001,train001_00003,W
4,-0.944100,153.803718,-552.651515,583.380258,5.572554,-50.273240,42.142277,1136.031773,0.308270,74.151517,...,65.573744,64.611180,64.396449,64.779543,2.372967,0.000273,0.349899,train001,train001_00004,W


Sleep_Stage
N2    33786
W     15828
N1     7753
R      7033
N3     2345
Name: count, dtype: int64


# Train Validation

In [ ]:
feature_cols = [c for c in train_table.columns if c not in ['subject_id', 'segment_id', TARGET]]
X = train_table[feature_cols]
y = train_table[TARGET]
groups = train_table['subject_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_valid = X.iloc[valid_idx]
y_valid = y.iloc[valid_idx]

rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=400,
        max_depth=18,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1
    ))
])

et = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', ExtraTreesClassifier(
        n_estimators=500,
        max_depth=22,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ))
])

ensemble = VotingClassifier(
    estimators=[('rf', rf), ('et', et)],
    voting='soft',
    n_jobs=-1
)

ensemble.fit(X_train, y_train)
valid_pred = ensemble.predict(X_valid)
valid_f1 = f1_score(y_valid, valid_pred, average='weighted')

print('Validation weighted F1 =', valid_f1)
print(classification_report(y_valid, valid_pred))


Validation weighted F1 = 0.42022996595037787
              precision    recall  f1-score   support

          N1       0.14      0.05      0.07      1547
          N2       0.52      0.79      0.63      6827
          N3       0.00      0.00      0.00       743
           R       0.11      0.02      0.03      1785
           W       0.47      0.44      0.45      3360

    accuracy                           0.49     14262
   macro avg       0.25      0.26      0.24     14262
weighted avg       0.39      0.49      0.42     14262



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Build Test Features

In [ ]:
def build_test_table(test_root: Path):
    rows = []
    test_files = sorted(test_root.glob('*/*.csv'))
    if not test_files:
        test_files = sorted(test_root.glob('*.csv'))

    print('num test segment files =', len(test_files))

    for file_path in test_files:
        df = pd.read_csv(file_path)
        df = add_signal_features(df)
        feat = summarize_segment(df, FEATURE_COLS)
        feat['id'] = file_path.stem
        rows.append(feat)

    return pd.DataFrame(rows)

test_table = build_test_table(test_root)
print('test_table shape =', test_table.shape)
display(test_table.head())


num test segment files = 7832
test_table shape = (7832, 91)


,BVP_mean,BVP_std,BVP_min,BVP_max,BVP_median,BVP_q25,BVP_q75,BVP_range,BVP_diff_mean,BVP_diff_std,...,ACC_MAG_std,ACC_MAG_min,ACC_MAG_max,ACC_MAG_median,ACC_MAG_q25,ACC_MAG_q75,ACC_MAG_range,ACC_MAG_diff_mean,ACC_MAG_diff_std,id
0,2.436631,97.086599,-502.470288,412.153790,8.922093,-37.086008,48.175610,914.624078,-0.078326,48.283056,...,0.287330,63.054035,65.904473,64.494097,64.353620,64.717858,2.850438,0.001008,0.334107,test001_00000
1,-0.224025,52.915803,-158.916642,71.545492,15.325783,-33.858390,42.803203,230.462134,-0.009085,26.038014,...,0.202147,63.834727,65.127188,64.687181,64.525993,64.834057,1.292462,0.000206,0.221378,test001_00001
2,0.453667,47.670564,-136.080170,68.509758,13.493460,-31.190312,38.141540,204.589928,0.092263,23.456152,...,0.185925,64.046817,65.121797,64.726649,64.568603,64.844648,1.074980,0.000905,0.210543,test001_00002
3,-0.237769,49.376062,-118.424662,61.473943,13.961334,-35.452339,41.395458,179.898605,0.170509,23.710986,...,0.198529,64.036383,65.126598,64.732661,64.573842,64.839409,1.090214,0.000626,0.225781,test001_00003
4,0.406600,62.424253,-421.064892,282.361400,10.670237,-31.745293,36.496230,703.426292,-0.103522,37.743450,...,0.267276,63.108991,66.193821,64.491814,64.379454,64.580246,3.084830,-0.001248,0.329934,test001_00004


# Final Fit and Submission

In [ ]:
final_model = VotingClassifier(
    estimators=[('rf', rf), ('et', et)],
    voting='soft',
    n_jobs=-1
)

final_model.fit(X, y)

test_pred = final_model.predict(test_table[feature_cols])

submission = sample_sub.copy()
pred_map = dict(zip(test_table['id'], test_pred))
submission['labels'] = submission['id'].map(pred_map)

missing_ids = submission['labels'].isna().sum()
print('missing ids =', missing_ids)
if missing_ids > 0:
    raise ValueError('Some submission ids were not matched with test segment files')

submission.to_csv(OUTPUT_PATH, index=False)
print('Saved submission to', OUTPUT_PATH)
display(submission.head(15))


missing ids = 0
Saved submission to /kaggle/working/submission.csv


,id,labels
0,test001_00000,W
1,test001_00001,N2
2,test001_00002,N2
3,test001_00003,N2
4,test001_00004,W
5,test001_00005,W
6,test001_00006,W
7,test001_00007,W
8,test001_00008,W
9,test001_00009,W
